# torchnative on Linux — does the published wheel compute?

The `manylinux_2_17_x86_64` wheel is on PyPI and its symbols resolve, but nothing
has ever executed on it: this machine has no Linux and no container runtime, so
the README marks that row ⚠️ rather than ✅.

This notebook is the missing run. **Runtime → Run all**, then the result block at
the bottom is what gets read back.

Two things make it more than `pip install`:

* the wheel is `cp313-abi3`, so it needs **Python ≥ 3.13** — Colab may be older
* this distribution **provides** `torch`; Colab already has one, and there is only
  one `torch` per interpreter

Both are handled by building a separate 3.13 environment rather than touching
Colab's own.


In [ ]:
# 0. What we are actually standing on.
import platform, sys, subprocess
print("platform :", platform.platform())
print("machine  :", platform.machine())
print("python   :", sys.version.split()[0])
print("libc     :", platform.libc_ver())


In [ ]:
# 1. A Python 3.13 that is not Colab's, via uv (no root, no apt).
!pip -q install uv 2>/dev/null | tail -1
!uv python install 3.13
!uv venv --python 3.13 /content/tn-venv
!/content/tn-venv/bin/python -c "import sys; print('venv python:', sys.version.split()[0])"


In [ ]:
# 2. The published wheel. Not a local build -- this is what a user gets.
!/content/tn-venv/bin/python -m pip -q install --upgrade pip
!/content/tn-venv/bin/python -m pip install "torchnative==0.0.9a0"


In [ ]:
# 3. Which wheel actually landed, and is it ours?
!/content/tn-venv/bin/python - <<'EOF'
import importlib.metadata as m, torch, sys
print("torchnative :", m.version("torchnative"))
print("torch       :", torch.__version__)
print("torch file  :", torch.__file__)
print("is the shim :", hasattr(torch._C, "_aten_implemented"))
print("aten ops    :", len(torch._C._aten_implemented()))
EOF


In [ ]:
# 4. The actual question: does it compute, and does it agree with known values?
#    Every expected number here was produced on macOS arm64 by the same source.
!/content/tn-venv/bin/python - <<'EOF'
import torch

checks = []
def check(name, got, want):
    ok = got == want
    checks.append(ok)
    print(f"{'PASS' if ok else 'FAIL'}  {name:38s} got={got!r}")
    if not ok:
        print(f"      {'':38s} want={want!r}")

a = torch.ones(2, 3)
b = torch.ones(3, 4)
check("mm sum",            (a @ b).sum().item(),                24.0)
check("mm shape",          tuple((a @ b).shape),                (2, 4))

lin = torch.nn.Linear(3, 4, bias=False)
with torch.no_grad():
    lin.weight.fill_(1.0)
    out = lin(torch.ones(1, 3))
check("nn.Linear sum",     out.sum().item(),                    12.0)

# Mixed-dtype promotion, released in 0.0.9a0 -- value as well as dtype.
x = torch.tensor([2049], dtype=torch.int64)
y = torch.tensor([1.0],  dtype=torch.float16)
check("sub(i64,f16) value", torch.sub(x, y).tolist(),           [2047.0])
check("sub(i64,f16) dtype", str(torch.sub(x, y).dtype),         "torch.float16")
check("eq(i64,f32) exact",  torch.eq(torch.tensor([16777217]),
                                     torch.tensor([16777216.0])).tolist(), [True])
f32 = torch.tensor([0.1]); f64 = torch.tensor([0.1], dtype=torch.float64)
check("cat promotes",       str(torch.cat([f32, f64]).dtype),   "torch.float64")

print()
print("RESULT:", "ALL PASS" if all(checks) else f"{checks.count(False)} FAILED", f"({len(checks)} checks)")
EOF


In [ ]:
# 5. A real model, which is the whole point -- real transformers, not a toy.
!/content/tn-venv/bin/python -m pip -q install "transformers>=5,<6"
!/content/tn-venv/bin/python - <<'EOF'
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer
name = "HuggingFaceTB/SmolLM2-135M"
m = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32); m.eval()
tok = AutoTokenizer.from_pretrained(name)
inp = tok("On-device inference is", return_tensors="pt")
with torch.no_grad():
    m.generate(**inp, max_new_tokens=4, do_sample=False, use_cache=True)
    t0 = time.perf_counter()
    o = m.generate(**inp, max_new_tokens=24, do_sample=False, use_cache=True)
    dt = time.perf_counter() - t0
text = tok.decode(o[0], skip_special_tokens=True)
print("generated:", repr(text))
print(f"speed    : {dt*1000:.1f} ms for 24 tokens -> {24/dt:.1f} tok/s")
print()
print("macOS arm64 produced, from the same source:")
print("  'On-device inference is a very powerful technique for learning from data. It is a'")
print("MATCH:", text.startswith("On-device inference is a very powerful technique"))
EOF


In [ ]:
# 6. Load-time quantisation, released in 0.0.9a0.
!/content/tn-venv/bin/python - <<'EOF'
import torch, resource
from transformers import AutoModelForCausalLM
from torchnative.quant import TorchnativeConfig
name = "HuggingFaceTB/SmolLM2-135M"
m = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32,
                                         quantization_config=TorchnativeConfig("q8_0"))
kinds = {}
for mod in m.modules():
    kinds[type(mod).__name__] = kinds.get(type(mod).__name__, 0) + 1
print("QuantizedLinear:", kinds.get("QuantizedLinear", 0), " Linear:", kinds.get("Linear", 0))
print("peak RSS       :", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6, 1), "MB")
print("macOS arm64: QuantizedLinear=210 Linear=1, peak 924 MB")
EOF


## What to read back

Cells 3–6. The lines that matter are `is the shim`, `RESULT:`, `MATCH:` and the
module counts. A `FAIL` is a finding, not a problem with the notebook — this run
exists precisely because nobody knows the answer yet.
